In [ ]:
import pandas as pd
import numpy as np
import ast
import os
import shutil
base = os.path.dirname(os.path.abspath('.'))
base = base + '/orca_descriptors'

def load_table(path):
    path = str(path)
    if path.lower().endswith(('.csv',)):
        return pd.read_csv(path)
    elif path.lower().endswith(('.xls', '.xlsx')):
        return pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported extension for {path}. Use .csv, .xls, or .xlsx")


def describe_col_type(series):
    s = series.rename('v')
    non_null = s.dropna()
    if non_null.empty:
        return 'x_nan'  # only nan/None values

    # numeric conversion check
    numeric = pd.to_numeric(non_null, errors='coerce')
    is_numeric = numeric.notna().all()

    if is_numeric:
        # check integerness of numeric values
        if np.all(np.floor(numeric) == numeric):
            return 'x_int'
        else:
            return 'x'

    # non-numeric at all -> string category/label
    return 'x_str'


def make_descriptor_encoding(desc_df, key='SMILES'):
    enc = {}
    x_i = 0
    x_str_j = 0
    x_nan_k = 0
    x_int_m = 0

    for col in desc_df.columns:
        if col == key:
            continue
        col_type = describe_col_type(desc_df[col])

        if col_type == 'x':
            enc[col] = f'x{x_i}'
            x_i += 1
        elif col_type == 'x_str':
            enc[col] = f'x_str{x_str_j}'
            x_str_j += 1
        elif col_type == 'x_nan':
            enc[col] = f'x_nan{x_nan_k}'
            x_nan_k += 1
        elif col_type == 'x_int':
            enc[col] = f'x_int{x_int_m}'
            x_int_m += 1
        else:
            enc[col] = f'x_unknown{col}'

    return enc


def merge_descriptor_target(desc_path, target_path, key_target='SMILES', key_descriptor='SMILES'):
    desc_df = load_table(desc_path)
    target_df = load_table(target_path)

    if key_descriptor not in desc_df.columns:
        raise KeyError(f"Key '{key_descriptor}' not found in descriptor file")
    if key_target not in target_df.columns:
        raise KeyError(f"Key '{key_target}' not found in target file")

    desc_smiles = set(desc_df[key_descriptor].dropna().astype(str))
    target_smiles = set(target_df[key_target].dropna().astype(str))
    missing_from_desc = sorted(target_smiles - desc_smiles)
    if missing_from_desc:
        print(f"Warning: {len(missing_from_desc)} SMILES entries in target are missing from descriptor file and will be dropped in merged_df:")
        for s in missing_from_desc[:20]:
            print(f"  - {s}")
        if len(missing_from_desc) > 20:
            print(f"  ... and {len(missing_from_desc)-20} more")

    merged = pd.merge(desc_df, target_df, left_on=key_descriptor, right_on=key_target, how='inner')
    encoding = make_descriptor_encoding(desc_df, key=key_descriptor)

    return merged, encoding

In [ ]:
### add the path to the descriptor and target files, and the desired output directory for ROBERT input
output_dir = 'ROBERT/test_splitseven'  # e.g. 'ROBERT/test_splitstratified'
split = 'even'     # options:    'even': splits the data evenly into training and test sets. (default)
                  #             'rnd': randomly splits the data.
                  #             'stratified': splits the data while preserving the distribution of the target variable. 
                  #             'KN': uses a k-means approach to select representative samples for training (good for intrapolation, bad for extrapolation). 
                  #             'extra_q1': selects the 20% lowest values.
                  #             'extra_q5': selects the 20% highest values.

### do you want to overwrite existing ROBERT input in the output directory??
overwrite = True
if not overwrite and os.path.exists(f'{output_dir}/CURATE'):
    print(f"Error: {output_dir}/CURATE already exists. \nSet overwrite=True to overwrite existing CURATE folder or choose a different output directory.")


In [ ]:
merged_df, encoding = merge_descriptor_target('20260422_LNiH2_library_001_to_download_2026_03_24_descriptors_.csv', 
                                              'experiment_data_ee.xlsx',
                                              key_target='INCHI', key_descriptor='inchi')


### encode the column names to the format expected by ROBERT, and save the csv for ROBERT input
encoding_ = {k: v for k, v in encoding.items() if k != 'SMILES'}
decoding  = {v: k for k, v in encoding.items()}
renamed_df = merged_df.rename(columns=encoding_)
renamed_df.drop(columns=['INCHI', 'yield', 'ee', 'Unnamed: 3'], inplace=True)

## save the csv for ROBERT input
print(f"Saving ROBERT input to: {output_dir}/input.csv")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

renamed_df.rename(columns={"x_str0": "SMILES"}, inplace=True)
renamed_df.drop(columns=['inchi', 'x_str1', 'x_str2'], inplace=True)
renamed_df.to_csv(f'{output_dir}/input.csv', index=False)
renamed_df.head()

In [ ]:
### This code used ROBERT to curate the data
### ref: https://robert.readthedocs.io/en/latest/index.html
from robert import curate as cur
curation = cur.curate(  csv_name=f'{output_dir}/input.csv',
                        y = 'ΔΔG',
                        names = 'SMILES',
                        )

### just some boring data cleaning and file moving to get the CURATE output into the desired output directory, and to avoid overwriting existing CURATE output if it already exists in the output directory
if overwrite and os.path.exists(f'{output_dir}/CURATE'):
    shutil.rmtree(f"{output_dir}/CURATE")
    print(f"Overwritten: {output_dir}/CURATE")
    shutil.move("CURATE", f"{output_dir}")  # destination option in curate() is currently broken, so we move the CURATE folder manually after curation is done
elif not os.path.exists(f'{output_dir}/CURATE'):
    print(f"CURATE output will be saved to: {output_dir}/CURATE")
    shutil.move("CURATE", f"{output_dir}")  # destination option in curate() is currently broken, so we move the CURATE folder manually after curation is done
else:
    print(f"Error: {output_dir}/CURATE already exists. \nSet overwrite=True to overwrite existing CURATE folder or choose a different output directory.")



In [ ]:
### this code used ROBERT to generate models (linear and non linear)
### ref: https://robert.readthedocs.io/en/latest/index.html
from robert import generate as gen 
### use the curated csv from ROBERT to generate models
gen.generate(csv_name=f'{output_dir}/CURATE/input_CURATE.csv', 
            y = 'ΔΔG',
            names = 'SMILES',
            split = split,
            pfi_epochs=5,
            kfold=3,
            seed=1)

### just some boring data cleaning and file moving to get the GENERATE output into the desired output directory, and to avoid overwriting existing GENERATE output if it already exists in the output directory
if overwrite and os.path.exists(f'{output_dir}/GENERATE'):
    shutil.rmtree(f"{output_dir}/GENERATE")
    print(f"Overwritten: {output_dir}/GENERATE")
    shutil.move("GENERATE", f"{output_dir}")  # destination option in generate() is currently broken, so we move the GENERATE folder manually after generation is done
elif not os.path.exists(f'{output_dir}/GENERATE'):
    print(f"GENERATE output will be saved to: {output_dir}/GENERATE")
    shutil.move("GENERATE", f"{output_dir}")  # destination option in generate() is currently broken, so we move the GENERATE folder manually after generation is done
else:
    print(f"Error: {output_dir}/GENERATE already exists. \nSet overwrite=True to overwrite existing GENERATE folder or choose a different output directory.")


In [ ]:
### verify function using ROBERT
from robert import verify as verify
verify.verify(params_dir = f'{output_dir}/GENERATE/Best_model/')

### just some boring data cleaning and file moving to get the VERIFY output into the desired output directory, and to avoid overwriting existing VERIFY output if it already exists in the output directory
if overwrite and os.path.exists(f'{output_dir}/VERIFY'):
    shutil.rmtree(f"{output_dir}/VERIFY")
    print(f"Overwritten: {output_dir}/VERIFY")
    shutil.move("VERIFY", f"{output_dir}")  # destination option in verify() is currently broken, so we move the VERIFY folder manually after verification is done
elif not os.path.exists(f'{output_dir}/VERIFY'):
    print(f"VERIFY output will be saved to: {output_dir}/VERIFY")
    shutil.move("VERIFY", f"{output_dir}")  # destination option in verify() is currently broken, so we move the VERIFY folder manually after verification is done
else:
    print(f"Error: {output_dir}/VERIFY already exists. \nSet overwrite=True to overwrite existing VERIFY folder or choose a different output directory.")


In [ ]:
### predict function using ROBERT - check performance on test set
from robert import predict as predict
pred = predict.predict(params_dir = f'{output_dir}/GENERATE/Best_model/')

### just some boring data cleaning and file moving to get the PREDICT output into the desired output directory, and to avoid overwriting existing PREDICT output if it already exists in the output directory
if overwrite and os.path.exists(f'{output_dir}/PREDICT'):
    shutil.rmtree(f"{output_dir}/PREDICT")
    print(f"Overwritten: {output_dir}/PREDICT")
    shutil.move("PREDICT", f"{output_dir}")  # destination option in predict() is currently broken, so we move the PREDICT folder manually after prediction is done
elif not os.path.exists(f'{output_dir}/PREDICT'):
    print(f"PREDICT output will be saved to: {output_dir}/PREDICT")
    shutil.move("PREDICT", f"{output_dir}")  # destination option in predict() is currently broken, so we move the PREDICT folder manually after prediction is done
else:
    print(f"Error: {output_dir}/PREDICT already exists. \nSet overwrite=True to overwrite existing PREDICT folder or choose a different output directory.")


In [ ]:
### report function using ROBERT - generate report from predictions
from robert import report as report
os.chdir(output_dir)
rep = report.report()
os.chdir(base)

In [ ]:
# decode the ROBERT output back to original column names

best_model_path = [f for f in os.listdir(f'{output_dir}/GENERATE/Best_model/PFI/') if f.endswith('PFI.csv')][0]  # get the name of the PFI csv file in the Best_model/PFI folder
df_best_model = pd.read_csv(f'{output_dir}/GENERATE/Best_model/PFI/{best_model_path}')
parameters = ast.literal_eval(df_best_model.loc[0, "X_descriptors"])

decoded_parameters = [decoding[param] for param in parameters]
print("Decoded parameters used in the best model:")
for param, decoded in zip(parameters, decoded_parameters):
    print(f"{param} -> {decoded}")